### ```__call__```

A class with ```__call__``` method makes its instances callable. I.e. the call ```x(a,b, ...)``` will result in calling this special method with the given parameters

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, x):
        return x * self.factor

double = Multiplier(2)
print(double(5))     # → 10     ← calling the INSTANCE like a function!

```double(5)``` secretly runs ```double.__call__(5)```. This is how you make an object behave like a function — useful for things that need to remember state between calls (here, factor).

## "Remember State Between Calls" — Simply

Let's contrast a plain function against `Multiplier` to see exactly what this phrase means.

---

### A Plain Function Has NO Memory Between Calls

```python
def multiply(x, y):
    return x * y

print(multiply(5, 2))    # → 10
print(multiply(5, 2))    # → 10   — you have to pass "2" AGAIN, every single time
print(multiply(7, 2))    # → 14   — and again
```

The function `multiply` **forgets everything** the instant it finishes running. It has no way to say *"remember that 2 from before — just use that again."* Every call starts completely fresh, with zero memory of any prior call.

---

### `Multiplier` — the Object REMEMBERS `factor`

```python
double = Multiplier(2)     # factor=2 is stored INSIDE double, ONCE
```

At this moment, `2` gets saved as `self.factor`, and it stays there — inside the `double` object — **forever** (until you delete `double` or change it):

```python
print(double.__dict__)     # → {'factor': 2}
```

Now every time you call `double(...)`, it **reaches back into that stored `factor`**, without you having to supply it again:

```python
print(double(5))    # → 10    (uses the REMEMBERED factor=2)
print(double(9))    # → 18    (STILL uses factor=2 — remembered from creation!)
print(double(100))   # → 200   (still factor=2)
```

**You never typed "2" again after the first line.** The object carries that value with it, across every subsequent call — that's the "state" being remembered.

---

### Side-by-Side — What Each Approach Requires You to Repeat

```python
# Plain function — you must supply BOTH values, every single time:
multiply(5, 2)
multiply(9, 2)
multiply(100, 2)
       ↑
   "2" repeated every call — the function has no memory of it

# Callable object — factor is set ONCE, then implicitly reused:
double = Multiplier(2)
double(5)
double(9)
double(100)
   ↑
"2" only appears ONCE, at creation — every later call just "remembers" it
```

---

### Where "State" Actually Lives — Back to `self.__dict__`

This connects directly to your earlier `__dict__` lesson! The "memory" isn't magic — it's literally the instance's attribute dictionary, persisting between calls:

```python
double = Multiplier(2)
print(double.__dict__)      # → {'factor': 2}    ← stored HERE

double(5)     # __call__ runs, reads self.factor from THIS dict
double(9)     # __call__ runs AGAIN, reads self.factor from the SAME dict — still 2!
```

Each call to `double(...)` is a **separate** execution of `__call__` — but they all read from the **same persistent `self.factor`**, because `self` is the **same object**, `double`, every time. That persistence across separate calls is exactly what "remembering state" means.

---

### Making Multiple Independent "Memories" — Where This Really Shines

The real payoff: you can create **several** callables, each remembering **its own separate** value:

```python
double = Multiplier(2)
triple = Multiplier(3)

print(double(5))    # → 10
print(triple(5))     # → 15    ← DIFFERENT remembered factor, same 'x' input!
```

```python
print(double.__dict__)   # → {'factor': 2}
print(triple.__dict__)    # → {'factor': 3}
```

Two **separate** dictionaries, two **separate** memories — even though both are built from the exact same `Multiplier` class and both respond to the same `(x)` call signature. A plain function couldn't do this trick at all — a function is a single, stateless piece of logic; it can't "fork" into multiple versions that each remember something different.

---

### A More Practical Example — A Running Counter

This makes the "remembering between calls" idea even more visible, since the state actually **changes** across calls:

```python
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count

c = Counter()
print(c())    # → 1
print(c())    # → 2     ← remembers it already ran once!
print(c())    # → 3     ← remembers it ran twice!
```

A plain function genuinely **cannot** do this — there's no way for an ordinary `def counter(): ...` to know it's been called before, without some external variable to track it. The object's `self.count` **is** that tracking mechanism, living quietly between calls.

---

### The One-Sentence Summary

> "Remembering state between calls" means the object holds onto data (like `self.factor` or `self.count`) **in its own attribute dictionary**, persisting from one call to the next — so later calls can rely on information set earlier, without you having to pass it in again each time. A plain function has no such persistent storage; it starts fresh every call. This is exactly why `Multiplier(2)` only needs the `2` supplied once, at creation, rather than on every single multiplication. 🎯